# 🔧 Feature Engineering

## Objectif
Créer et tester des features avancées pour améliorer les performances des modèles:
- Features textuelles avancées
- Features linguistiques
- Features de style d'écriture
- Analyse d'importance des features

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.preprocessing import StandardScaler

# Modules locaux
from src.data_loader import load_training_data
from src.preprocessing import extract_full_text, clean_text
from src.feature_engineering import (
    count_hashtags, count_mentions, count_emojis,
    count_urls, has_media, create_all_features
)

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Imports réussis")

## 1. Chargement des Données

In [ ]:
# Charger les données
print("📥 Chargement des données...")
df_train = load_training_data('../data/train.jsonl')
print(f"✓ {len(df_train)} tweets chargés")

# Extraire le texte
df_train['full_text'] = df_train.apply(extract_full_text, axis=1)
print("✓ Texte extrait")

## 2. Features Textuelles de Base

In [ ]:
# Features de base
print("🔧 Création des features de base...")

df_train['text_length'] = df_train['full_text'].str.len()
df_train['word_count'] = df_train['full_text'].str.split().str.len()
df_train['avg_word_length'] = df_train['full_text'].apply(
    lambda x: np.mean([len(word) for word in str(x).split()]) if pd.notna(x) else 0
)

df_train['num_hashtags'] = df_train['full_text'].apply(count_hashtags)
df_train['num_mentions'] = df_train['full_text'].apply(count_mentions)
df_train['num_emojis'] = df_train['full_text'].apply(count_emojis)
df_train['num_urls'] = df_train['full_text'].apply(count_urls)

print("✓ Features de base créées")
df_train[['full_text', 'text_length', 'word_count', 'num_hashtags', 'num_mentions', 'label']].head()

## 3. Features Avancées

In [ ]:
# Features de ponctuation
print("🔧 Features de ponctuation...")

df_train['num_exclamation'] = df_train['full_text'].str.count('!')
df_train['num_question'] = df_train['full_text'].str.count('\?')
df_train['num_dots'] = df_train['full_text'].str.count('\.')

# Features de capitalisation
df_train['num_uppercase'] = df_train['full_text'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper())
)
df_train['ratio_uppercase'] = df_train['num_uppercase'] / (df_train['text_length'] + 1)

# Features de contenu
df_train['has_url'] = (df_train['num_urls'] > 0).astype(int)
df_train['has_hashtag'] = (df_train['num_hashtags'] > 0).astype(int)
df_train['has_mention'] = (df_train['num_mentions'] > 0).astype(int)

print("✓ Features avancées créées")

## 4. Analyse d'Importance des Features

In [ ]:
# Sélectionner les features numériques
feature_cols = [
    'text_length', 'word_count', 'avg_word_length',
    'num_hashtags', 'num_mentions', 'num_emojis', 'num_urls',
    'num_exclamation', 'num_question', 'num_dots',
    'num_uppercase', 'ratio_uppercase',
    'has_url', 'has_hashtag', 'has_mention'
]

# Agréger par utilisateur (prendre la moyenne)
user_features = df_train.groupby('challenge_id').agg({
    **{col: 'mean' for col in feature_cols},
    'label': 'first'
}).reset_index()

X = user_features[feature_cols].fillna(0)
y = user_features['label']

print(f"✓ {len(X)} utilisateurs avec {len(feature_cols)} features")

In [ ]:
# Calculer l'information mutuelle
print("📊 Calcul de l'importance des features...")

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_scores = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

# Visualisation
plt.figure(figsize=(10, 8))
mi_scores.plot(kind='barh', color='skyblue')
plt.title('Importance des Features (Mutual Information)', fontsize=14, fontweight='bold')
plt.xlabel('Mutual Information Score')
plt.ylabel('Feature')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n📈 Top 10 Features:")
print(mi_scores.head(10))

## 5. Corrélations entre Features

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(12, 10))
correlation_matrix = X.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matrice de Corrélation des Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Features fortement corrélées
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            high_corr.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr:
    print("\n⚠️ Features fortement corrélées (|r| > 0.8):")
    for feat1, feat2, corr in high_corr:
        print(f"  - {feat1} ↔ {feat2}: {corr:.3f}")
else:
    print("\n✓ Pas de corrélations excessives détectées")

## 6. Distribution des Features par Classe

In [ ]:
# Sélectionner les top features
top_features = mi_scores.head(6).index.tolist()

# Visualisation
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    user_features[user_features['label'] == 0][feature].hist(
        bins=30, alpha=0.6, label='Observer', ax=axes[i], color='skyblue'
    )
    user_features[user_features['label'] == 1][feature].hist(
        bins=30, alpha=0.6, label='Influencer', ax=axes[i], color='salmon'
    )
    axes[i].set_title(f'Distribution: {feature}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Fréquence')
    axes[i].legend()

plt.tight_layout()
plt.show()

## 7. Test avec Modèle Simple

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Créer un pipeline simple
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

# Test avec différents ensembles de features
feature_sets = {
    'Top 5': mi_scores.head(5).index.tolist(),
    'Top 10': mi_scores.head(10).index.tolist(),
    'Toutes': feature_cols
}

print("🧪 Test des ensembles de features:\n")
results = {}

for name, features in feature_sets.items():
    X_subset = user_features[features].fillna(0)
    scores = cross_val_score(pipeline, X_subset, y, cv=5, scoring='accuracy')
    results[name] = scores.mean()
    print(f"{name:12} : {scores.mean():.4f} (+/- {scores.std():.4f})")

# Visualisation
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values(), color=['skyblue', 'lightgreen', 'salmon'])
plt.title('Performance selon l\'Ensemble de Features', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (CV)')
plt.ylim(0, 1)
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (50%)')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Conclusions

### Insights:
1. **Features importantes**: Les features les plus discriminantes identifiées
2. **Corrélations**: Features redondantes à potentiellement éliminer
3. **Performance**: Impact de la sélection de features sur la performance

### Prochaines étapes:
- Combiner ces features avec TF-IDF
- Tester sur des modèles plus complexes
- Feature engineering additionnel (sentiment, NER, etc.)
- Optimisation des hyperparamètres

In [ ]:
# Sauvegarder les features pour utilisation future
print("💾 Sauvegarde des features pour utilisation future...")
user_features.to_csv('../data/user_features.csv', index=False)
print("✓ Features sauvegardées dans data/user_features.csv")
print("\n📝 Passez au notebook 04_advanced_models.ipynb")